In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from  langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

/tmp/ipykernel_11691/4166537714.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from dotenv import load_dotenv

load_dotenv()


True

In [5]:
import  pandas as pd
books  = pd.read_csv("/media/zunayed/HDD_code2/book-recommender/data/books_cleadned.csv")

In [6]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5083    9788172235222 On A Train Journey Home To North...
5084    9788173031014 This book tells the tale of a ma...
5085    9788179921623 Wisdom to Create a Life of Passi...
5086    9788185300535 This collection of the timeless ...
5087    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5088, dtype: str

In [7]:
books["tagged_description"].to_csv("tagged_description.txt", sep = "\n", index=False, header=False)

In [8]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(
    chunk_size=1,
    chunk_overlap=0,
    separator="\n"
)
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1168, which is longer than the specified 1
Created a chunk of size 1214, which is longer than the specified 1
Created a chunk of size 373, which is longer than the specified 1
Created a chunk of size 309, which is longer than the specified 1
Created a chunk of size 483, which is longer than the specified 1
Created a chunk of size 482, which is longer than the specified 1
Created a chunk of size 960, which is longer than the specified 1
Created a chunk of size 188, which is longer than the specified 1
Created a chunk of size 843, which is longer than the specified 1
Created a chunk of size 296, which is longer than the specified 1
Created a chunk of size 197, which is longer than the specified 1
Created a chunk of size 881, which is longer than the specified 1
Created a chunk of size 1088, which is longer than the specified 1
Created a chunk of size 1189, which is longer than the specified 1
Created a chunk of size 304, which is longer than the specified 1
Create

In [23]:
import os
import shutil
import time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "chroma_db"

print("Initializing local HuggingFace embeddings...")
local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

BATCH_SIZE = 500
total_docs = len(documents)
print(f"Starting local ingestion of {total_docs} documents...\n")

all_texts = []
all_metadatas = []
all_embeddings = []

for i in range(0, total_docs, BATCH_SIZE):
    batch_docs = documents[i:i + BATCH_SIZE]
    texts_batch = [doc.page_content for doc in batch_docs]
    metadatas_batch = [doc.metadata for doc in batch_docs]

    embeddings_batch = local_embeddings.embed_documents(texts_batch)

    all_texts.extend(texts_batch)
    all_metadatas.extend(metadatas_batch)
    all_embeddings.extend(embeddings_batch)

    print(f"Processed local vectors for Docs {i} to {min(i + BATCH_SIZE, total_docs)}...")

# Step 2: Feed the pre-calculated raw vectors directly into Chroma
print("\n--- Generation Complete! Feeding vectors to local storage... ---")

if os.path.exists(PERSIST_DIR):
    print(f"Clearing existing database at '{PERSIST_DIR}' for a clean run...")
    shutil.rmtree(PERSIST_DIR)

# 1. Instantiate the Chroma client pointing to your directory
db_books = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=local_embeddings
)

# 2. Add the pre-computed text strings and embeddings directly into the collection
db_books.add_texts(
    texts=all_texts,
    embeddings=all_embeddings,
    metadatas=all_metadatas
)

print("\n Ingestion completely finished! Your full library is successfully vectorized and stored on disk.")

Initializing local HuggingFace embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Starting local ingestion of 5088 documents...

Processed local vectors for Docs 0 to 500...
Processed local vectors for Docs 500 to 1000...
Processed local vectors for Docs 1000 to 1500...
Processed local vectors for Docs 1500 to 2000...
Processed local vectors for Docs 2000 to 2500...
Processed local vectors for Docs 2500 to 3000...
Processed local vectors for Docs 3000 to 3500...
Processed local vectors for Docs 3500 to 4000...
Processed local vectors for Docs 4000 to 4500...
Processed local vectors for Docs 4500 to 5000...
Processed local vectors for Docs 5000 to 5088...

--- Generation Complete! Feeding vectors to local storage... ---


InternalError: Database error: error returned from database: (code: 14) unable to open database file

In [25]:
import os
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Use a BRAND NEW absolute folder name to bypass any hidden Jupyter file locks
PERSIST_DIR = os.path.abspath("my_local_chromadb")

if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

# Explicitly create the empty directory so SQLite doesn't panic
os.makedirs(PERSIST_DIR, exist_ok=True)

print("Initializing local HuggingFace embeddings on CPU...")
local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(f"Starting automatic local ingestion of {len(documents)} documents...")
print("Please wait a minute or two for your CPU to process everything...")

# 2. Let LangChain natively handle all the vectorization and storage!
# No manual loops needed since we aren't restricted by API rate limits anymore.
db_books = Chroma.from_documents(
    documents=documents,
    embedding=local_embeddings,
    persist_directory=PERSIST_DIR
)

print("\n✅ Ingestion completely finished! Your local database is ready.")

Initializing local HuggingFace embeddings on CPU...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Starting automatic local ingestion of 5088 documents...
Please wait a minute or two for your CPU to process everything...

✅ Ingestion completely finished! Your local database is ready.


In [26]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='cb387352-e48b-4cb3-b1f1-e7e9cb82778a', metadata={'source': 'tagged_description.txt'}, page_content='9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.'),
 Document(id='c1137629-41fb-4ec8-a211-c7ecb22d6c75', metadata={'source': 'tagged_description.txt'}, page_content="9780786808380 Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing the baby to some basic -- and sometimes playful -- information about the subjects."),
 Document(id='940abd9d-0ae6-46cf-8965-cc61ea50e411', metadata={'source': '

In [27]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3664,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [28]:
def semantic_recommendations(
        query: str,
        top_k: int = 10,

) -> pd.DataFrame:
    recs = db_books.similarity_search(query,  k = 50)
    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)].head(top_k)



In [29]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)
    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [30]:
retrieve_semantic_recommendations("A book for children ")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
88,9780060004507,0060004509,An Old-Fashioned Thanksgiving,Louisa May Alcott;James Bernardin,Juvenile Fiction,http://books.google.com/books/content?id=SKYr4...,An adaptation of the original story follows th...,2005.0,3.70,32.0,599.0,An Old-Fashioned Thanksgiving,9780060004507 An adaptation of the original st...
264,9780060882600,0060882603,The Annotated Charlotte's Web,E. B. White,Juvenile Nonfiction,http://books.google.com/books/content?id=vaYYH...,"Charlotte's Web, one of America's best-loved c...",2006.0,4.16,320.0,41.0,The Annotated Charlotte's Web,"9780060882600 Charlotte's Web, one of America'..."
279,9780060915186,0060915188,An American Childhood,Annie Dillard,Biography & Autobiography,http://books.google.com/books/content?id=tRihT...,A book that instantly captured the hearts of r...,1988.0,3.91,255.0,7086.0,An American Childhood,9780060915186 A book that instantly captured t...
331,9780060976118,006097611X,Operation Wandering Soul,Richard Powers,Fiction,http://books.google.com/books/content?id=nIGIm...,"Highly imaginative and emotionally powerful, t...",1994.0,3.62,352.0,366.0,Operation Wandering Soul,9780060976118 Highly imaginative and emotional...
399,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
401,9780064403870,0064403874,"R-T, Margaret, and the Rats of NIMH",Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=WTHHH...,"When Margaret and her younger brother, Artie, ...",1991.0,3.52,272.0,631.0,"R-T, Margaret, and the Rats of NIMH",9780064403870 When Margaret and her younger br...
402,9780064404419,0064404412,The Rainbow People,Laurence Yep,Juvenile Fiction,http://books.google.com/books/content?id=5AHwq...,"""Culled from 69 stories collected in a [1930s]...",1992.0,3.75,208.0,202.0,The Rainbow People,"9780064404419 ""Culled from 69 stories collecte..."
406,9780064405713,0064405710,Crazy Lady!,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=aJwh7...,Increasingly alienated from his widowed father...,1995.0,3.64,192.0,1481.0,Crazy Lady!,9780064405713 Increasingly alienated from his ...
410,9780064406925,006440692X,Winter on the Farm,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=IvlKH...,The Little House books tell the story of a lit...,1997.0,4.13,32.0,400.0,Winter on the Farm,9780064406925 The Little House books tell the ...
562,9780140139976,0140139974,Sailor Song,Ken Kesey,Fiction,http://books.google.com/books/content?id=-pPSO...,"In Alaska to film a famous children's book, th...",1993.0,3.57,533.0,1956.0,Sailor Song,9780140139976 In Alaska to film a famous child...
